## 定义模型

In [3]:
# from langchain.agents import create_agent
# from langchain.chat_models import init_chat_model
# from dotenv import load_dotenv
# import os
# load_dotenv()

# model = init_chat_model(
#     model="qwen3.5-plus",
#     model_provider="openai",
#     base_url=os.getenv(DASHSCOPE_BASE_URL),
#     api_key=os.getenv(DASHSCOPE_API_KEY)
# )
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent

load_dotenv()


model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# agent = create_agent(model=model)

## 定义工具

In [4]:
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

tavily=TavilySearch(
    max_results=5,
    topic="general"
)
@tool
def web_search(query: str):
    """根据关键词搜索互联网"""
    return tavily.invoke(query)

## 添加记忆管理

In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

connection = sqlite3.connect("resources/personal_chef.db", check_same_thread=False)
checkpointer= SqliteSaver(connection)
checkpointer.setup()

## 定义智能体

In [6]:
from langchain.agents import create_agent

system_prompt="""
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份 “当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以 “可用食材清单” 为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

## 测试

In [7]:
from langchain_core.messages import HumanMessage, AIMessage

multimodel_messages= HumanMessage([
    {
        "type": "text", "text": "帮我看看能做什么"
    },
    {
        "type": "image", "url": "https://pic.rmb.bdstatic.com/bjh/bc117c78c118/250321/3e2503fe8b412a0e7c470d9cf828f4d5.jpeg"
    }
])

config = {"configurable": {"thread_id": "5"}}
response = agent.invoke({
    "messages": [multimodel_messages]
}, config)

for message in response['messages']:
    message.pretty_print()

def get_message(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""
    """根据thread_id查询checkpoint"""
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})
    if not cp:
        return []
    """安全获取messages"""
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []
    messages = channel_values.get("messages", [])
    if not messages:
        return []
    
    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue
        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})
    return result;

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://pic.rmb.bdstatic.com/bjh/bc117c78c118/250321/3e2503fe8b412a0e7c470d9cf828f4d5.jpeg'}]
================================== Ai Message ==================================
Tool Calls:
  web_search (call_dd3d03918df445b5afb52442)
 Call ID: call_dd3d03918df445b5afb52442
  Args:
    query: 葡萄 苹果 李子 橙子 鸡蛋 小番茄 能做什么菜谱
================================= Tool Message =================================
Name: web_search

{"query": "葡萄 苹果 李子 橙子 鸡蛋 小番茄 能做什么菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.reddit.com/r/EatCheapAndHealthy/comments/xzxcc4/i_have_a_bunch_of_cherry_tomatoes_what_do_i_do/?tl=zh-hans", "title": "我有一堆小番茄，怎么办啊？有什么好吃的做法推荐吗？ - Reddit", "content": "把它们扔进橄榄油里，加盐和胡椒，再加点蒜，然后在300度烤到它们开始起泡。可以做成cottage cheese、牛油果吐司或意面的配料。", "score": 0.4391465, "raw_content": null}, {"url": "https://www.yout

In [8]:
get_message("5")

[{'role': 'user',
  'content': [{'type': 'text', 'text': '帮我看看能做什么'},
   {'type': 'image',
    'url': 'https://pic.rmb.bdstatic.com/bjh/bc117c78c118/250321/3e2503fe8b412a0e7c470d9cf828f4d5.jpeg'}]},
 {'role': 'assistant',
  'content': '好的，已为您识别冰箱食材并检索了相关食谱。您的冰箱里水果种类非常丰富，还有鸡蛋和一些蔬菜（小番茄）。\n\n以下是为您整理的 **“当前可用食材清单”** 及 **精选食谱建议报告**：\n\n### 1. 当前可用食材清单 (Current Available Ingredients)\n\n*   **水果类:** 葡萄 (青/红), 苹果, 李子, 橙子, 梨, 猕猴桃 (可能), 百香果 (可能)\n*   **蔬菜类:** 小番茄 (圣女果)\n*   **蛋类:** 鸡蛋 (一整盒，约10-12枚)\n*   **其他:** 一些包装好的食材（因包装遮挡无法完全确认，暂不计入）\n\n---\n\n### 2. 精选食谱建议报告 (Recipe Recommendation Report)\n\n根据您的食材，我为您设计了以下三款食谱，按推荐度排序：\n\n#### **🥇 推荐一：意式小番茄烘蛋 (Frittata with Cherry Tomatoes)**\n\n*   **使用食材:** 鸡蛋、小番茄\n*   **营养得分:** ⭐⭐⭐⭐⭐ (高蛋白、富含维生素)\n*   **制作难度:** ⭐⭐ (简单)\n*   **综合得分:** **95/100**\n*   **推荐理由:** 这是一道经典快手菜，能完美消耗大量的小番茄和鸡蛋。口感丰富，营养均衡，非常适合作为早午餐或轻晚餐。\n*   **参考图片:**\n\n![意式小番茄烘蛋](https://th.bing.com/th/id/OIP.8q6J5vJ5vJ5vJ5vJ5vJ5vJ5vJ5?pid=ImgDet&rs=1)\n\n*   **简要步骤:**\n    1.  小番茄对半切开。\n    2.  鸡

## 清空回话历史

In [ ]:
checkpointer.delete_thread("5")